<a href="https://colab.research.google.com/github/yogendra785/suspicious-login-detection-pytorch/blob/main/Suspicious_Login_Behavior_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random


np.random.seed(42)
random.seed(42)

def generate_login_data(num_samples=5000):
    data = []

    for _ in range(num_samples):

        is_suspicious = 1 if random.random() < 0.2 else 0

        if is_suspicious == 0:
            # NORMAL LOGIN BEHAVIOR
            distance_km = random.uniform(0, 50) # Same city, typical commute
            time_since_last_min = random.uniform(120, 2880) # A few hours to a couple of days
            failed_attempts = random.choice([0, 0, 0, 1]) # Might mistype a password once
            device_type = 0 # 0 = Known device

        else:
            # SUSPICIOUS LOGIN BEHAVIOR (e.g., "Impossible Travel" or Brute Force)
            scenario = random.choice(['impossible_travel', 'brute_force'])

            if scenario == 'impossible_travel':
                distance_km = random.uniform(1000, 10000) # Halfway across the world
                time_since_last_min = random.uniform(1, 15) # But only 5 minutes since last login!
                failed_attempts = random.choice([0, 1])
                device_type = random.choice([0, 1])

            elif scenario == 'brute_force':
                distance_km = random.uniform(0, 500)
                time_since_last_min = random.uniform(1, 60)
                failed_attempts = random.randint(5, 15) # Hammering the login button
                device_type = 1 # 1 = Unknown device

        data.append([distance_km, time_since_last_min, failed_attempts, device_type, is_suspicious])

    columns = ['distance_from_last_login_km', 'time_since_last_login_min',
               'failed_attempts_24h', 'is_unknown_device', 'is_suspicious']

    return pd.DataFrame(data, columns=columns)


df = generate_login_data(5000)

# Round the float columns for a cleaner look
df['distance_from_last_login_km'] = df['distance_from_last_login_km'].round(2)
df['time_since_last_login_min'] = df['time_since_last_login_min'].round(2)


print("🚀 Dataset Snapshot:")
display(df.head())

print("\n📊 Class Distribution (0 = Normal, 1 = Suspicious):")
print(df['is_suspicious'].value_counts())

🚀 Dataset Snapshot:


,distance_from_last_login_km,time_since_last_login_min,failed_attempts_24h,is_unknown_device,is_suspicious
0,1.25,879.08,0,0,0
1,7090.30,13.49,0,1,1
2,2967.74,8.07,0,0,1
3,35.07,1277.87,1,0,0
4,40.47,137.94,0,0,0



📊 Class Distribution (0 = Normal, 1 = Suspicious):
is_suspicious
0    3938
1    1062
Name: count, dtype: int64


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   distance_from_last_login_km  5000 non-null   float64
 1   time_since_last_login_min    5000 non-null   float64
 2   failed_attempts_24h          5000 non-null   int64  
 3   is_unknown_device            5000 non-null   int64  
 4   is_suspicious                5000 non-null   int64  
dtypes: float64(2), int64(3)
memory usage: 195.4 KB


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X=df.drop('is_suspicious',axis=1).values
y=df['is_suspicious'].values

X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

#Defining our custom pytorch datasets

class LoginDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.FloatTensor(features)
    self.labels=torch.FloatTensor(labels).unsqueeze(1)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self,idx):
    return self.features[idx],self.labels[idx]

train_dataset=LoginDataset(X_train_scaled,y_train)
test_dataset=LoginDataset(X_test_scaled,y_test)

BATCH_SIZE=64
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader= DataLoader(test_dataset, batch_size=BATCH_SIZE,shuffle=False)


print("✅ Data successfully converted to PyTorch Tensors!")
print(f"Number of training batches: {len(train_loader)}")

# Let's peek at exactly what one batch looks like
sample_features, sample_labels = next(iter(train_loader))
print(f"\nShape of one batch of features: {sample_features.shape} -> (Batch Size, Number of Features)")
print(f"Shape of one batch of labels: {sample_labels.shape} -> (Batch Size, Label)")



✅ Data successfully converted to PyTorch Tensors!
Number of training batches: 63

Shape of one batch of features: torch.Size([64, 4]) -> (Batch Size, Number of Features)
Shape of one batch of labels: torch.Size([64, 1]) -> (Batch Size, Label)
